# 2. Graph Extraction (Pure GraphRAG)

Notebook ini mengekstrak entitas dan relasi dari raw EMR menggunakan LLM, lalu menyimpannya ke Neo4j.

In [1]:
import os
import sys
import pandas as pd
import time

sys.path.append(os.path.abspath('..'))

from src.graph.client import GraphClient
from src.config import settings
from src.ingestion.extractor import LLMGraphExtractor
from src.services.providers import get_embeddings

import logging
logging.basicConfig(level=logging.INFO)

d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
client = GraphClient(
    uri=settings.neo4j_uri,
    user=settings.neo4j_user,
    password=settings.neo4j_password
)
extractor = LLMGraphExtractor(temperature=0.0)

INFO:src.graph.client:Connected to Neo4j at bolt://localhost:7687


In [ ]:
file_path = '../data/Dashboard EMR(report1776669858353).csv'
try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    try:
        df = pd.read_csv(file_path, encoding='latin1')
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
df_sample = df.head(20) # Hapus .head(5) untuk jalankan semua

In [4]:
def ingest_extraction_to_neo4j(extraction, emr_name, client):
    if not extraction: return
    client.run_query("MERGE (e:EMRRecord {emr_name: $name})", {"name": emr_name})
    for node in extraction.nodes:
        query = f"""
        MERGE (n:{node.label} {{name: $name}})
        WITH n MATCH (e:EMRRecord {{emr_name: $emr_name}})
        MERGE (e)-[:MENTIONS]->(n)
        """
        client.run_query(query, {"name": node.name, "emr_name": emr_name})
    for rel in extraction.relationships:
        rel_query = f"""
        MATCH (s {{name: $source}}) MATCH (t {{name: $target}}) MERGE (s)-[:{rel.type}]->(t)
        """
        try: client.run_query(rel_query, {"source": rel.source, "target": rel.target})
        except Exception: pass

In [5]:
start_time = time.time()
for idx, row in df_sample.iterrows():
    emr_name = str(row.get('emr_name', f'EMR-UNK-{idx}'))
    text_chunk = f"""
    EMR Record ID: {emr_name}\nMachine Model: {row.get('Machine Model', '')}\nComponent: {row.get('Techcare Component', '')} | Sub-Component: {row.get('Techcare Sub Component', '')}\nSymptom/Problem: {row.get('Symptom', '')}\nCause of Problem: {row.get('Caused of Problem', '')}\nAction Taken: {row.get('Action (How was problem corrected?)', '')}\nPart Replaced: {row.get('Part Description', '')} (Part No: {row.get('Main Cause Part No', '')})\n    """
    print(f"Extracting EMR {emr_name}...")
    extraction = extractor.extract(text_chunk)
    if extraction:
        ingest_extraction_to_neo4j(extraction, emr_name, client)
print(f"\nCompleted in {time.time() - start_time:.2f} seconds.")

Extracting EMR EMR-UNK-0...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Extracting EMR EMR-UNK-1...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Extracting EMR EMR-UNK-2...


...

Extracting EMR EMR-UNK-3...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Extracting EMR EMR-UNK-4...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



Completed in 269.90 seconds.


## Generate Embeddings untuk Entity Resolution
Agar kita bisa melakukan deteksi kesamaan pada *Notebook 3*, setiap entitas harus memiliki representasi *vector embedding*.

In [6]:
print("Men-generate Vector Embeddings...")
embedder = get_embeddings()
labels_to_embed = ["SymptomPattern", "ProblemCluster", "RootCausePattern", "ActionPattern"]

for label in labels_to_embed:
    nodes = client.run_query(f"MATCH (n:{label}) WHERE n.embedding IS NULL RETURN n.name AS name")
    names = [n['name'] for n in nodes if n['name']]
    if names:
        print(f"Embedding {len(names)} {label}...")
        embeddings = embedder.embed_documents(names)
        for name, emb in zip(names, embeddings):
            client.run_query(f"MATCH (n:{label} {{name: $name}}) SET n.embedding = $emb", {"name": name, "emb": emb})
print("Embeddings selesai!")

Men-generate Vector Embeddings...


INFO:src.services.embedding_service:Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2 ...


Embedding 5 SymptomPattern...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/s

Embedding 631 RootCausePattern...


Batches: 100%|██████████| 20/20 [00:01<00:00, 11.50it/s]


Embedding 941 ActionPattern...


Batches: 100%|██████████| 30/30 [00:02<00:00, 12.52it/s]


Embeddings selesai!


## Deterministic Enrichment (Post-Processing)
Langkah ini memastikan bahwa seluruh **Techcare Component** dan **ActionPattern** dari CSV disuntikkan secara deterministik dan terhubung dengan benar ke graf, menutupi kelemahan LLM yang mungkin melewatkan entitas tersebut.

In [ ]:
import re

def normalize_name(text: str) -> str:
    text = str(text).strip()
    return re.sub(r'\s+', ' ', text).lower()

print("Memulai deterministic enrichment untuk Component dan ActionPattern...")
count_comp, count_action = 0, 0

for idx, row in df_sample.iterrows():
    emr_name = str(row.get('EMR Name', row.get('emr_name', f'EMR-UNK-{idx}'))).strip()
    
    # 1. Enrich Component
    comp = str(row.get('Techcare Component', '')).strip()
    sub_comp = str(row.get('Techcare Sub Component', '')).strip()
    
    if comp and comp.lower() != 'nan':
        client.run_query("MERGE (c:Component {name: $name})", {"name": comp})
        client.run_query("MATCH (e:EMRRecord {emr_name: $emr}), (c:Component {name: $comp}) MERGE (e)-[:HAS_COMPONENT]->(c)", {"emr": emr_name, "comp": comp})
        count_comp += 1
        
        if sub_comp and sub_comp.lower() != 'nan':
            client.run_query("MERGE (c:Component {name: $name})", {"name": sub_comp})
            client.run_query("MATCH (e:EMRRecord {emr_name: $emr}), (c:Component {name: $sub}) MERGE (e)-[:HAS_COMPONENT]->(c)", {"emr": emr_name, "sub": sub_comp})
            client.run_query("MATCH (sub:Component {name: $sub_name}), (main:Component {name: $main_name}) MERGE (sub)-[:PART_OF]->(main)", {"sub_name": sub_comp, "main_name": comp})
            
    # 2. Enrich Action Pattern
    action_raw = str(row.get('Action (How was problem corrected?)', '')).strip()
    if action_raw and action_raw.lower() != 'nan':
        action_name = normalize_name(action_raw)
        client.run_query("MERGE (a:ActionPattern {name: $name})", {"name": action_name})
        client.run_query("MATCH (e:EMRRecord {emr_name: $emr}), (a:ActionPattern {name: $act}) MERGE (e)-[:MENTIONS]->(a)", {"emr": emr_name, "act": action_name})
        client.run_query("MATCH (e:EMRRecord {emr_name: $emr})-[:MENTIONS]->(s:SymptomPattern), (a:ActionPattern {name: $act}) MERGE (s)-[:RESOLVED_BY]->(a)", {"emr": emr_name, "act": action_name})
        client.run_query("MATCH (e:EMRRecord {emr_name: $emr})-[:MENTIONS]->(rc:RootCausePattern), (a:ActionPattern {name: $act}) MERGE (rc)-[:RESOLVED_BY]->(a)", {"emr": emr_name, "act": action_name})
        count_action += 1

print(f"Enrichment selesai: {count_comp} Component dan {count_action} ActionPattern ditambahkan ke graf secara deterministik.")


In [7]:
client.close()